# MALTO Recruitment Hackathon 2026 — Text Authorship Detection

**GitHub Repository:** [Sajjad-Shahali/text-authorship-detection](https://github.com/Sajjad-Shahali/text-authorship-detection)

---

## Competition Overview

| | |
|---|---|
| **Competition** | MALTO Recruitment Hackathon 2026 |
| **Task** | 6-class text authorship classification |
| **Metric** | Macro F1 Score (all classes weighted equally) |
| **Training set** | 2,400 samples |
| **Test set** | 600 samples |
| **Constraint** | No pre-trained AI detectors, no external data |
| **Best Public LB** | **0.92422** |
| **Best CV (5-fold)** | **0.9393** |

| Label | Source | Train Samples | Train % |
|-------|--------|:---:|:---:|
| 0 | Human | 1520 | 63.3% |
| 1 | DeepSeek | 80 | 3.3% |
| 2 | Grok | 160 | 6.7% |
| 3 | Claude | 80 | 3.3% |
| 4 | Gemini | 240 | 10.0% |
| 5 | ChatGPT | 320 | 13.3% |

---

## This Notebook — Run 10 Complete Reproduction

This notebook is a **complete, self-contained training and inference pipeline** that
reproduces **Run 10** (`2026-03-16_221821_run`), the best-performing run.

All custom classes are defined inline — no `src/` directory required.
The notebook trains the model from scratch (including 5-fold CV and threshold optimization)
and generates a submission CSV.

### Full Pipeline

```
Raw text
  └─► Preprocessor (unicode normalization, whitespace)
        └─► FeatureUnion (~100k+ dimensions)
              ├─ Word TF-IDF       [50k, ngram (1,2), word]
              ├─ Char TF-IDF       [50k, ngram (2,6), char_wb]
              ├─ Stylometric       [43 hand-crafted features]
              └─ Function-word TF-IDF [151 fixed vocabulary words]
                    └─► StackingClassifier (3-fold internal CV)
                          ├─ Base 1: TwoStageClassifier
                          │    ├─ 6-class LR (C=0.5, balanced)
                          │    └─ DS/Grok binary LR (C=1.5, balanced)
                          ├─ Base 2: TfidfMLPClassifier
                          │    └─ TruncatedSVD(500) → MLP(512→256, ReLU)
                          └─ Base 3: LGBMTfidfClassifier
                               └─ TruncatedSVD(300) → LightGBM(300 trees)
                                    └─► Meta: LogisticRegression(C=0.1, balanced)
                                          └─► Per-class threshold scaling
                                                └─► DS/Grok pair ratio threshold
```

### Key Design Choices

| Choice | Reason |
|--------|--------|
| No lowercasing | Capitalization is a style signal (Claude uses **bold**, Gemini uses headers) |
| Preserve punctuation | Comma/semicolon rates distinguish LLMs statistically |
| Balanced class weights | DeepSeek (3.3%) would be invisible without reweighting |
| Stacking over voting | Meta-LR learns per-model confidence, not just average |
| Per-class thresholds | Corrects calibration error on minority classes |
| DS/Grok pair threshold | Ratio threshold is more stable than absolute probability |

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────────
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

try:
    import lightgbm as lgb
    print(f'lightgbm {lgb.__version__} ✓')
except ImportError:
    _install('lightgbm>=4.0.0')

try:
    import sklearn
    print(f'scikit-learn {sklearn.__version__} ✓')
except ImportError:
    _install('scikit-learn>=1.3.0')

In [ ]:
# ── Standard imports ───────────────────────────────────────────────────────────
import math
import os
import re
import json
import string
import unicodedata
import warnings
from datetime import datetime
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin, clone
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import StackingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import MaxAbsScaler
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight

from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
np.random.seed(42)

print('All imports OK ✓')

## 1. Paths & Configuration (Run 10 — Frozen)

These are the **exact** hyperparameters from Run 10 (`2026-03-16_221821_run`).
Source: `artifacts/experiments/2026-03-16_221821_run/config_snapshot.yaml`

Full source code: [github.com/Sajjad-Shahali/text-authorship-detection](https://github.com/Sajjad-Shahali/text-authorship-detection)

In [ ]:
# ── Environment detection: Kaggle vs local ────────────────────────────────────
if os.path.exists('/kaggle/input'):
    DATA_DIR   = Path('/kaggle/input/malto-hackathon')
    OUTPUT_DIR = Path('/kaggle/working')
    print('Environment: Kaggle')
else:
    REPO_ROOT  = Path(r'D:/hachaton/text-authorship-detection')
    DATA_DIR   = REPO_ROOT / 'data'
    OUTPUT_DIR = REPO_ROOT / 'artifacts/submissions'
    print('Environment: Local')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_FILE = DATA_DIR / 'train.csv'
TEST_FILE  = DATA_DIR / 'test.csv'

print(f'DATA_DIR   : {DATA_DIR}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}')
print(f'train.csv  : {TRAIN_FILE.exists()}')
print(f'test.csv   : {TEST_FILE.exists()}')

In [ ]:
# ── Run 10 frozen configuration ────────────────────────────────────────────────
# Exact values from: artifacts/experiments/2026-03-16_221821_run/config_snapshot.yaml

CONFIG = {
    # ── Preprocessing (preserve style signals)
    'preprocessing': {
        'normalize_unicode':      True,
        'strip_whitespace':       True,
        'remove_repeated_spaces': True,
        'lowercase':              False,  # KEEP: capitalization = style signal
        'remove_punctuation':     False,  # KEEP: punctuation rates are discriminative
        'remove_numbers':         False,  # KEEP: numeric patterns carry signal
    },
    # ── Feature engineering (4 branches)
    'features': {
        'word_tfidf': {
            'analyzer': 'word', 'ngram_range': (1, 2),
            'max_features': 50000, 'min_df': 3, 'max_df': 0.95, 'sublinear_tf': True,
        },
        'char_tfidf': {
            'analyzer': 'char_wb', 'ngram_range': (2, 6),
            'max_features': 50000, 'min_df': 2, 'max_df': 0.95, 'sublinear_tf': True,
        },
        'stylometric':         {'enabled': True},          # 43 hand-crafted features
        'function_word_tfidf': {'enabled': True, 'sublinear_tf': True},  # 151 words
    },
    # ── Cross-validation
    'validation': {'n_splits': 5, 'random_state': 42, 'shuffle': True},
    # ── LightGBM hyperparameters
    'lgbm': {
        'n_svd_components': 300, 'n_estimators': 300, 'num_leaves': 31,
        'learning_rate': 0.05, 'min_child_samples': 10,
        'subsample': 0.8, 'colsample_bytree': 0.8,
        'reg_alpha': 0.1, 'reg_lambda': 0.1,
    },
    # ── MLP hyperparameters
    'mlp': {
        'n_svd_components': 500, 'hidden_layer_sizes': (512, 256),
        'alpha': 0.01, 'max_iter': 300,
        'early_stopping': True, 'validation_fraction': 0.1,
    },
    # ── Logistic regression (base + meta)
    'logistic_regression_balanced': {'C': 0.5, 'max_iter': 1000, 'solver': 'lbfgs'},
    # ── TwoStage trigger settings
    'two_stage_margin_gap':    0.40,
    'two_stage_ds_threshold':  0.52,
    # ── Random seed
    'random_state': 42,
}

CLASS_NAMES  = {0: 'Human', 1: 'DeepSeek', 2: 'Grok', 3: 'Claude', 4: 'Gemini', 5: 'ChatGPT'}
N_CLASSES    = 6
RANDOM_STATE = CONFIG['random_state']

lc = CONFIG['lgbm']
mc = CONFIG['mlp']
print('Run 10 configuration loaded.')
print(f"  Feature branches  : word_tfidf, char_tfidf, stylometric (43), function_word_tfidf (151 words)")
print(f"  LGBM              : SVD({lc['n_svd_components']}) + {lc['n_estimators']} trees, {lc['num_leaves']} leaves")
print(f"  MLP               : SVD({mc['n_svd_components']}) + layers {mc['hidden_layer_sizes']}")
print(f"  CV folds          : {CONFIG['validation']['n_splits']}")

## 2. Data Loading

In [ ]:
# ── Load competition data ──────────────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_FILE)
test_df  = pd.read_csv(TEST_FILE)

X_train  = train_df['text'].fillna('').astype(str).tolist()
y_train  = train_df['label'].values
X_test   = test_df['text'].fillna('').astype(str).tolist()
test_ids = test_df['id'].values

print(f'Train: {len(X_train):,} samples')
print(f'Test : {len(X_test):,} samples')
print(f'\nClass distribution (train):')
for label in range(N_CLASSES):
    count = (y_train == label).sum()
    bar = '█' * (count // 25)
    print(f'  {CLASS_NAMES[label]:>8} [{label}]: {count:4d}  {bar}')

## 3. Custom Component Definitions

All pipeline components are defined inline so this notebook is fully self-contained.
Each class is identical to the production implementation in `src/` of the
[GitHub repository](https://github.com/Sajjad-Shahali/text-authorship-detection).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PREPROCESSOR  →  src/preprocess.py
#
# Philosophy: preserve ALL stylistic signals.
# - Capitalization: Claude/Gemini use **bold** and ## headers; DeepSeek does not.
# - Punctuation: comma/semicolon rates differ between GPT-4 and DeepSeek.
# - Newlines: markdown structure (bullets, headers) is a strong LLM fingerprint.
# ══════════════════════════════════════════════════════════════════════════════

class Preprocessor(BaseEstimator, TransformerMixin):
    """Minimal, style-preserving text preprocessor."""

    def __init__(self, normalize_unicode=True, strip_whitespace=True,
                 remove_repeated_spaces=True, lowercase=False,
                 remove_punctuation=False, remove_numbers=False):
        self.normalize_unicode      = normalize_unicode
        self.strip_whitespace       = strip_whitespace
        self.remove_repeated_spaces = remove_repeated_spaces
        self.lowercase              = lowercase
        self.remove_punctuation     = remove_punctuation
        self.remove_numbers         = remove_numbers

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        return [self._clean(t) for t in X]

    def _clean(self, text):
        if not isinstance(text, str):
            text = str(text) if text is not None else ''
        if self.normalize_unicode:
            text = unicodedata.normalize('NFC', text)
        if self.strip_whitespace:
            text = text.strip()
        if self.remove_repeated_spaces:
            text = re.sub(r' {2,}', ' ', text)
            text = re.sub(r'[\t\r\n]+', ' ', text)
            text = re.sub(r' {2,}', ' ', text)
        if self.lowercase:
            text = text.lower()
        if self.remove_punctuation:
            text = re.sub(r'[^\w\s]', '', text)
        if self.remove_numbers:
            text = re.sub(r'\d+', '<NUM>', text)
        return text

print('Preprocessor ✓')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FEATURE ENGINEERING  →  src/features.py
#
# 4 feature branches combined with sklearn FeatureUnion:
#   1. Word TF-IDF     — vocabulary + phrase style
#   2. Char TF-IDF     — punctuation, spacing, morphological fingerprints
#   3. Stylometric     — 43 hand-crafted features (this run)
#   4. Function-word   — topic-neutral style fingerprint (151 fixed words)
# ══════════════════════════════════════════════════════════════════════════════

# ── Function-word vocabulary: 151 topic-neutral style markers ─────────────────
FUNCTION_WORDS = [
    # Determiners
    'the','a','an','this','that','these','those','my','your','his','her','its','our','their',
    # Prepositions
    'of','in','on','at','to','for','with','by','from','about','as','into','through','during',
    'before','after','above','below','between','among','under','over','within','without','upon',
    # Coordinating conjunctions
    'and','but','or','so','yet',
    # Subordinating conjunctions
    'although','because','since','while','if','unless','until','when','where','whether','though',
    # Auxiliaries
    'is','are','was','were','be','been','being','have','has','had','do','does','did',
    'will','would','shall','should','may','might','must','can','could',
    # Personal pronouns
    'i','you','he','she','it','we','they','me','him','us','them',
    # Discourse / transition markers (strong LLM fingerprint)
    'however','therefore','moreover','furthermore','additionally',
    'nevertheless','nonetheless','thus','hence','consequently',
    'accordingly','whereas','meanwhile','subsequently','overall',
    'also','too','either','neither',
    # Degree / intensifiers
    'very','quite','rather','somewhat','even','just','only',
    'indeed','certainly','particularly','especially','generally',
    'actually','basically','literally','simply',
    # Quantifiers
    'all','some','any','each','every','many','much','few','little',
    'more','most','other','another','same','different','various','several',
    # Relative / interrogative
    'which','who','whom','whose','what','how','why',
    # Negation
    'not','no','never','nor',
]

print(f'Function-word vocabulary: {len(FUNCTION_WORDS)} words')


class DenseToSparse(BaseEstimator, TransformerMixin):
    """Convert dense numpy array to scipy sparse for FeatureUnion."""
    def fit(self, X, y=None): return self
    def transform(self, X, y=None):
        return sp.csr_matrix(X if isinstance(X, np.ndarray) else np.array(X))


class StyleometricTransformer(BaseEstimator, TransformerMixin):
    """
    43 hand-crafted stylometric features (Run 10 version).

    Groups:
      Length metrics (4): char count, word count, sentence count, paragraph count
      Lexical (5): TTR, avg word length, long-word ratio, very-short/long sentence ratios
      Paragraph (3): avg para length, words per sentence, words per paragraph
      Punctuation (11): comma, period, exclamation, question, colon, semicolon,
                         quote, parenthesis, ellipsis, dash, (sub-total=10 + apc=1)
      Char composition (5): uppercase ratio, digit ratio, caps-word ratio, newline ratio
      Markdown (8): bullet rate, numbered-list rate, header rate, code rate,
                     bold rate, italic rate, link rate, starts-with-I ratio
      Discourse (3): transition word rate, hedge word rate
      DS/Grok discriminators (7): punct variety, sent CV, transition rate,
                                   first-sent words, proper noun density,
                                   hedge rate, question per sent, sent range, log-length
    """
    _NUM_RE  = re.compile(r'^\s*\d+[\.)].+', re.MULTILINE)
    _HEAD_RE = re.compile(r'^#{1,6}\s', re.MULTILINE)
    _BOLD_RE = re.compile(r'\*\*')
    _ITAL_RE = re.compile(r'(?<!\*)\*(?!\*)')
    _CODE_RE = re.compile(r'`')
    _LINK_RE = re.compile(r'\[.+?\]\(.+?\)')
    _ELIP_RE = re.compile(r'\.\.\.')
    _CAPS_RE = re.compile(r'\b[A-Z]{2,}\b')
    _TRANS_RE = re.compile(
        r'\b(however|therefore|moreover|furthermore|additionally|consequently|'
        r'nevertheless|nonetheless|whereas|thus|hence|accordingly|subsequently|'
        r'in conclusion|in summary|for example|for instance|in particular)\b',
        re.IGNORECASE)
    _HEDGE_RE = re.compile(
        r'\b(may|might|could|possibly|perhaps|likely|suggests|suggest|'
        r'appears|appear|seems|seem|potentially|presumably|arguably|'
        r'apparently|typically|usually|often)\b',
        re.IGNORECASE)

    def fit(self, X, y=None): return self

    def transform(self, X, y=None):
        return np.array([self._f(t) for t in X], dtype=np.float32)

    def _f(self, text):
        """Extract exactly 43 features (Run 10 exact specification)."""
        if not text or not isinstance(text, str):
            return [0.0] * 43

        NL2, NL1 = '\n\n', '\n'
        ch    = len(text)
        words = text.split();  nw  = max(len(words), 1)
        sents = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
        ns    = max(len(sents), 1)
        paras = [p.strip() for p in text.split(NL2) if p.strip()]
        np_   = max(len(paras), 1)
        lines = text.split(NL1);  nl_ = max(len(lines), 1)

        wl  = [len(w.strip(string.punctuation)) for w in words if w.strip(string.punctuation)]
        awl = float(np.mean(wl)) if wl else 0.0
        ttr = len(set(w.lower().strip(string.punctuation) for w in words)) / nw
        lwr = sum(1 for l in wl if l > 6) / nw
        swc = [len(s.split()) for s in sents]
        vss = sum(1 for c in swc if c < 5)  / ns
        vls = sum(1 for c in swc if c > 30) / ns
        apc = float(np.mean([len(p) for p in paras])) if paras else 0.0

        cm  = text.count(',')   / nw
        per = text.count('.')   / nw
        ex  = text.count('!')   / nw
        qu  = text.count('?')   / nw
        co  = text.count(':')   / nw
        se  = text.count(';')   / nw
        qt  = (text.count('"') + text.count("'") +
               text.count('\u201c') + text.count('\u201d')) / nw
        pa  = (text.count('(') + text.count(')')) / nw
        el  = len(self._ELIP_RE.findall(text)) / nw
        da  = (text.count('-') + text.count('\u2014')) / nw

        alpha = [c for c in text if c.isalpha()]
        na    = max(len(alpha), 1)
        ucr   = sum(1 for c in alpha if c.isupper()) / na
        dr    = sum(1 for c in text if c.isdigit()) / max(ch, 1)
        cwr   = len(self._CAPS_RE.findall(text)) / nw
        nlr   = text.count(NL1) / max(ch, 1)

        br  = sum(1 for l in lines if l.lstrip().startswith(('-', '*', '\u2022'))) / nl_
        nr  = len(self._NUM_RE.findall(text))  / nl_
        hr  = len(self._HEAD_RE.findall(text)) / nl_
        cr  = len(self._CODE_RE.findall(text)) / max(ch, 1) * 100
        bor = len(self._BOLD_RE.findall(text)) / max(ch, 1) * 100
        ir  = len(self._ITAL_RE.findall(text)) / max(ch, 1) * 100
        lkr = len(self._LINK_RE.findall(text)) / nw
        isr = sum(1 for s in sents if re.match(r'^I\s', s.strip())) / ns

        punct_chars   = [c for c in text if c in string.punctuation]
        punct_variety = len(set(punct_chars)) / max(len(punct_chars), 1)
        sent_cv       = float(np.std(swc) / max(np.mean(swc), 1)) if len(swc) > 1 else 0.0
        trans_rate    = len(self._TRANS_RE.findall(text)) / ns
        fsw           = float(len(sents[0].split())) if sents else 0.0
        cap_words     = sum(1 for w in words if w and w[0].isupper())
        proper_noun_dens = max(cap_words - ns, 0) / nw
        hedge_rate    = len(self._HEDGE_RE.findall(text)) / ns
        qps           = text.count('?') / ns
        sent_range    = float(max(swc) - min(swc)) if len(swc) > 1 else 0.0
        text_len_log  = math.log10(max(ch, 1))

        # ── Exactly 43 features (Run 10) ──────────────────────────────────────
        return [
            ch, nw, ns, np_, awl, nw/ns, nw/np_, ttr, lwr,          # 9
            vss, vls, apc, cm, per, ex, qu, co, se, qt, pa, el, da,  # 13
            ucr, dr, cwr, nlr, br, nr, hr, cr, bor, ir, lkr, isr,   # 12
            punct_variety, sent_cv, trans_rate, fsw, proper_noun_dens,  # 5
            hedge_rate, qps, sent_range, text_len_log,               # 4
        ]  # total = 43


class StyleometricPipeline(BaseEstimator, TransformerMixin):
    """StyleometricTransformer + MaxAbsScaler + sparse conversion."""

    _FEATURE_NAMES = [
        'ch','nw','ns','np','awl','nw_per_s','nw_per_p','ttr','lwr',
        'vss','vls','apc','cm','per','ex','qu','co','se','qt','pa','el','da',
        'ucr','dr','cwr','nlr','br','nr','hr','cr','bor','ir','lkr','isr',
        'punct_variety','sent_cv','trans_rate','first_sent_words',
        'proper_noun_density','hedge_rate','question_per_sent','sent_range','text_len_log',
    ]

    def __init__(self):
        self.extractor = StyleometricTransformer()
        self.scaler    = MaxAbsScaler()

    def fit(self, X, y=None):
        self.scaler.fit(self.extractor.transform(X))
        return self

    def transform(self, X, y=None):
        return sp.csr_matrix(self.scaler.transform(self.extractor.transform(X)))

    def get_feature_names_out(self, input_features=None):
        return np.array(self._FEATURE_NAMES)


def build_feature_union(cfg: Dict) -> FeatureUnion:
    """Build FeatureUnion: word_tfidf + char_tfidf + stylometric + function_word_tfidf."""
    fc = cfg.get('features', {})
    wc = fc.get('word_tfidf', {})
    cc = fc.get('char_tfidf', {})
    fw = fc.get('function_word_tfidf', {})

    word_tfidf = TfidfVectorizer(
        analyzer=wc.get('analyzer', 'word'),
        ngram_range=tuple(wc.get('ngram_range', [1, 2])),
        max_features=wc.get('max_features', 50000),
        min_df=wc.get('min_df', 3),
        max_df=wc.get('max_df', 0.95),
        sublinear_tf=wc.get('sublinear_tf', True),
        strip_accents='unicode',
        token_pattern=r'(?u)\b\w+\b',
    )
    char_tfidf = TfidfVectorizer(
        analyzer=cc.get('analyzer', 'char_wb'),
        ngram_range=tuple(cc.get('ngram_range', [2, 6])),
        max_features=cc.get('max_features', 50000),
        min_df=cc.get('min_df', 2),
        max_df=cc.get('max_df', 0.95),
        sublinear_tf=cc.get('sublinear_tf', True),
    )
    func_word_tfidf = TfidfVectorizer(
        analyzer='word',
        vocabulary=FUNCTION_WORDS,
        ngram_range=(1, 1),
        sublinear_tf=fw.get('sublinear_tf', True),
        token_pattern=r'(?u)\b\w+\b',
    )
    return FeatureUnion(transformer_list=[
        ('word_tfidf',         word_tfidf),
        ('char_tfidf',         char_tfidf),
        ('stylometric',        StyleometricPipeline()),
        ('function_word_tfidf', func_word_tfidf),
    ])


print('Feature engineering ✓ (43 stylometric features, 151 function words)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODEL CLASSES  →  src/models.py
# ══════════════════════════════════════════════════════════════════════════════

class TwoStageClassifier(ClassifierMixin, BaseEstimator):
    """
    Two-stage classifier that sharpens the DeepSeek / Grok boundary.

    Stage 1: full 6-class LogisticRegression (C=0.5, balanced weights).
    Stage 2: binary DeepSeek-vs-Grok LogisticRegression (C=1.5, balanced),
             triggered only when BOTH DS and Grok appear in the top-2 predicted
             classes (top2_trigger=True) AND their probability gap is small
             (margin_trigger_gap=0.40).

    In predict_proba: when stage-2 fires, the DS/Grok probability mass is
    redistributed according to the binary classifier's confidence, while
    leaving all other class probabilities unchanged.
    """
    _estimator_type = 'classifier'
    DEEPSEEK = 1
    GROK     = 2

    def __init__(self, base_classifier=None, binary_classifier=None,
                 margin_trigger_gap=None, binary_ds_threshold=0.5, top2_trigger=False):
        self.base_classifier     = base_classifier
        self.binary_classifier   = binary_classifier
        self.margin_trigger_gap  = margin_trigger_gap
        self.binary_ds_threshold = binary_ds_threshold
        self.top2_trigger        = top2_trigger

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.base_clf_ = clone(self.base_classifier) if self.base_classifier is not None else \
            LogisticRegression(C=0.5, max_iter=1000, solver='lbfgs',
                               class_weight='balanced', random_state=42)
        self.base_clf_.fit(X, y)
        mask = (y == self.DEEPSEEK) | (y == self.GROK)
        if mask.sum() < 4:
            self.binary_clf_ = None
            return self
        self.binary_clf_ = clone(self.binary_classifier) if self.binary_classifier is not None else \
            LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs',
                               class_weight=None, random_state=42)
        self.binary_clf_.fit(X[mask], y[mask])
        return self

    def _trigger_mask(self, X, preds, proba=None):
        base_ambig = (preds == self.DEEPSEEK) | (preds == self.GROK)
        if proba is None:
            return base_ambig
        conditions = [base_ambig]
        if self.top2_trigger:
            sorted_idx = np.argsort(proba, axis=1)[:, -2:]
            conditions.append(np.all(np.isin(sorted_idx, [self.DEEPSEEK, self.GROK]), axis=1))
        if self.margin_trigger_gap is not None:
            margin = np.abs(proba[:, self.DEEPSEEK] - proba[:, self.GROK])
            conditions.append(margin < self.margin_trigger_gap)
        result = conditions[0]
        for c in conditions[1:]:
            result = result & c
        return result

    def _apply_binary(self, X_sub):
        bp = self.binary_clf_.predict_proba(X_sub)
        bc = self.binary_clf_.classes_
        if self.DEEPSEEK not in bc or self.GROK not in bc:
            return self.binary_clf_.predict(X_sub)
        ds_pos = int(np.where(bc == self.DEEPSEEK)[0][0])
        return np.where(bp[:, ds_pos] >= self.binary_ds_threshold, self.DEEPSEEK, self.GROK)

    def predict(self, X):
        proba = self.base_clf_.predict_proba(X)
        preds = np.argmax(proba, axis=1).copy()
        if self.binary_clf_ is None:
            return preds
        mask = self._trigger_mask(X, preds, proba)
        if mask.any():
            preds[mask] = self._apply_binary(X[mask])
        return preds

    def predict_proba(self, X):
        proba = self.base_clf_.predict_proba(X).copy()
        if self.binary_clf_ is None or not hasattr(self.binary_clf_, 'predict_proba'):
            return proba
        preds = np.argmax(proba, axis=1)
        mask  = self._trigger_mask(X, preds, proba)
        if mask.any():
            bp = self.binary_clf_.predict_proba(X[mask])
            bc = self.binary_clf_.classes_
            if self.DEEPSEEK in bc and self.GROK in bc:
                ds_pos = int(np.where(bc == self.DEEPSEEK)[0][0])
                gk_pos = int(np.where(bc == self.GROK)[0][0])
                for i, idx in enumerate(np.where(mask)[0]):
                    total = proba[idx, self.DEEPSEEK] + proba[idx, self.GROK]
                    if total > 0:
                        proba[idx, self.DEEPSEEK] = total * bp[i, ds_pos]
                        proba[idx, self.GROK]     = total * bp[i, gk_pos]
        return proba


class TfidfMLPClassifier(ClassifierMixin, BaseEstimator):
    """
    TruncatedSVD(n) → MLPClassifier.

    Rationale: MLPClassifier cannot handle 100k-dim sparse inputs efficiently.
    TruncatedSVD compresses to n_svd_components dense dimensions (~70% variance
    retained at 500 components), giving the MLP a tractable dense input.
    Non-linear boundaries capture feature interactions impossible for LR.
    Class imbalance handled via compute_sample_weight('balanced', y).
    """
    _estimator_type = 'classifier'

    def __init__(self, n_svd_components=500, hidden_layer_sizes=(512, 256),
                 activation='relu', alpha=0.01, max_iter=300,
                 early_stopping=True, validation_fraction=0.1,
                 random_state=42, deepseek_boost_factor=1.0, verbose=False):
        self.n_svd_components    = n_svd_components
        self.hidden_layer_sizes  = hidden_layer_sizes
        self.activation          = activation
        self.alpha               = alpha
        self.max_iter            = max_iter
        self.early_stopping      = early_stopping
        self.validation_fraction = validation_fraction
        self.random_state        = random_state
        self.deepseek_boost_factor = deepseek_boost_factor
        self.verbose             = verbose

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        n_components  = min(self.n_svd_components, X.shape[1] - 1, X.shape[0] - 1)
        self.svd_ = TruncatedSVD(n_components=n_components, random_state=self.random_state)
        X_dense = self.svd_.fit_transform(X)
        sample_weights = compute_sample_weight('balanced', y)
        self.mlp_ = MLPClassifier(
            hidden_layer_sizes=self.hidden_layer_sizes, activation=self.activation,
            solver='adam', alpha=self.alpha, max_iter=self.max_iter,
            early_stopping=self.early_stopping, validation_fraction=self.validation_fraction,
            random_state=self.random_state, n_iter_no_change=15,
        )
        self.mlp_.fit(X_dense, y, sample_weight=sample_weights)
        return self

    def predict(self, X):       return self.mlp_.predict(self.svd_.transform(X))
    def predict_proba(self, X): return self.mlp_.predict_proba(self.svd_.transform(X))


class LGBMTfidfClassifier(ClassifierMixin, BaseEstimator):
    """
    TruncatedSVD(n) → LightGBM gradient boosting.

    Gradient boosting (leaf-wise growth) over dense latent TF-IDF features.
    Orthogonal to MLP: tree splits handle feature interactions differently.
    Naturally handles class imbalance via per-sample balanced weights.
    """
    _estimator_type = 'classifier'

    def __init__(self, n_svd_components=300, n_estimators=300, num_leaves=31,
                 learning_rate=0.05, min_child_samples=10, subsample=0.8,
                 colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1, random_state=42):
        self.n_svd_components  = n_svd_components
        self.n_estimators      = n_estimators
        self.num_leaves        = num_leaves
        self.learning_rate     = learning_rate
        self.min_child_samples = min_child_samples
        self.subsample         = subsample
        self.colsample_bytree  = colsample_bytree
        self.reg_alpha         = reg_alpha
        self.reg_lambda        = reg_lambda
        self.random_state      = random_state

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.svd_ = TruncatedSVD(n_components=self.n_svd_components, random_state=self.random_state)
        X_dense = self.svd_.fit_transform(X)
        cw = compute_class_weight('balanced', classes=self.classes_, y=y)
        sw = np.array([cw[int(np.where(self.classes_ == yi)[0][0])] for yi in y])
        self.lgbm_ = LGBMClassifier(
            n_estimators=self.n_estimators, num_leaves=self.num_leaves,
            learning_rate=self.learning_rate, min_child_samples=self.min_child_samples,
            subsample=self.subsample, colsample_bytree=self.colsample_bytree,
            reg_alpha=self.reg_alpha, reg_lambda=self.reg_lambda,
            random_state=self.random_state, n_jobs=-1, verbose=-1,
        )
        self.lgbm_.fit(X_dense, y, sample_weight=sw)
        return self

    def predict(self, X):       return self.lgbm_.predict(self.svd_.transform(X))
    def predict_proba(self, X): return self.lgbm_.predict_proba(self.svd_.transform(X))


print('Model classes ✓ (TwoStageClassifier, TfidfMLPClassifier, LGBMTfidfClassifier)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# THRESHOLD OPTIMIZATION  →  src/threshold_optimizer.py
# ══════════════════════════════════════════════════════════════════════════════

def apply_thresholds(proba: np.ndarray, thresholds: np.ndarray) -> np.ndarray:
    """
    Per-class probability scaling followed by argmax.

    Formula:  ŷ = argmax_k ( p_k * s_k )

    where s_k is the learned scale factor for class k.
    s_k > 1.0 → model biased toward predicting class k (boost recall)
    s_k < 1.0 → model biased away from predicting class k (reduce false positives)
    """
    return np.argmax(proba * thresholds, axis=1)


def apply_ds_grok_pair_threshold(proba: np.ndarray, preds: np.ndarray,
                                  pair_threshold: float) -> np.ndarray:
    """
    DeepSeek / Grok pair ratio threshold.

    Applied after per-class scaling. Only fires when:
      - The prediction is DS or Grok (from per-class threshold step)
      - P(DS) + P(Grok) > 0.15  (strong DS/Grok signal present)

    Formula:
      ratio = P(DS) / (P(DS) + P(Grok))
      ŷ = DS   if ratio >= pair_threshold
      ŷ = Grok otherwise

    pair_threshold > 0.5 → conservative DeepSeek prediction
    Run 10 optimal: 0.55
    """
    preds = preds.copy()
    ds_grok_sum = proba[:, 1] + proba[:, 2]
    ambiguous = (ds_grok_sum > 0.15) & np.isin(preds, [1, 2])
    if ambiguous.any():
        ratio = proba[ambiguous, 1] / np.maximum(ds_grok_sum[ambiguous], 1e-9)
        preds[ambiguous] = np.where(ratio >= pair_threshold, 1, 2)
    return preds


def optimize_thresholds(oof_proba: np.ndarray, y_true: np.ndarray,
                         n_grid: int = 11, lo: float = 0.5, hi: float = 3.0) -> np.ndarray:
    """
    Grid-search per-class probability scale factors on out-of-fold predictions.

    Algorithm:
      1. Compute per-class F1 on unscaled OOF predictions
      2. Order classes by worst F1 first (DeepSeek typically worst at 0.77)
      3. For each class in order: grid-search scale in [lo, hi]
         (greedy coordinate descent — holds other scales fixed)
      4. Repeat for 2 passes to capture cross-class interactions

    Returns: array of shape (n_classes,) with optimal scale factors
    """
    n_classes  = oof_proba.shape[1]
    thresholds = np.ones(n_classes)
    grid       = np.linspace(lo, hi, n_grid)

    # Compute per-class F1 to set optimization order
    base_preds = np.argmax(oof_proba, axis=1)
    f1_per_class = []
    for c in range(n_classes):
        m  = (y_true == c)
        tp = ((base_preds == c) & m).sum()
        fp = ((base_preds == c) & ~m).sum()
        fn = ((base_preds != c) & m).sum()
        d  = 2*tp + fp + fn
        f1_per_class.append(2*tp/d if d > 0 else 0.0)
    order = np.argsort(f1_per_class)  # worst first

    for _ in range(2):  # two passes
        for c in order:
            best_f1, best_val = -1.0, thresholds[c]
            for s in grid:
                t = thresholds.copy()
                t[c] = s
                preds = apply_thresholds(oof_proba, t)
                f1 = f1_score(y_true, preds, average='macro', zero_division=0)
                if f1 > best_f1:
                    best_f1, best_val = f1, s
            thresholds[c] = best_val

    return thresholds


def optimize_ds_grok_threshold(oof_proba: np.ndarray, y_true: np.ndarray,
                                 thresholds: np.ndarray, n_grid: int = 21) -> float:
    """Grid-search DS/Grok pair ratio threshold in [0.25, 0.75]."""
    best_f1, best_t = -1.0, 0.5
    for t in np.linspace(0.25, 0.75, n_grid):
        p = apply_thresholds(oof_proba, thresholds)
        p = apply_ds_grok_pair_threshold(oof_proba, p, t)
        f1 = f1_score(y_true, p, average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t


print('Threshold optimization functions ✓')

In [ ]:
# ── Pipeline builder ───────────────────────────────────────────────────────────

def build_stacking_lgbm(cfg: Dict, seed: int = 42) -> StackingClassifier:
    """
    Build stacking_lgbm: [TwoStage + MLP + LGBM] → LR meta-learner.

    The 3 base estimators are deliberately orthogonal:
      TwoStage : sparse linear + dedicated DS/Grok binary refinement
      MLP      : dense non-linear (SVD latent space, 512→256 hidden)
      LGBM     : tree-based non-linear (complementary splits to MLP)

    StackingClassifier uses 3-fold internal CV to generate 18-dimensional
    meta-features (3 models × 6 class probabilities) for the 2400 training
    samples. The meta LogisticRegression (C=0.1, strongly regularized) learns
    the optimal trust weights per model per class.
    """
    cfg_lr = cfg.get('logistic_regression_balanced', {})
    ds_thr = cfg.get('two_stage_ds_threshold', 0.52)
    gap    = cfg.get('two_stage_margin_gap', 0.40)
    mc     = cfg.get('mlp', {})
    lc     = cfg.get('lgbm', {})

    base_two_stage = TwoStageClassifier(
        base_classifier=LogisticRegression(
            C=cfg_lr.get('C', 0.5), max_iter=cfg_lr.get('max_iter', 1000),
            solver=cfg_lr.get('solver', 'lbfgs'), class_weight='balanced',
            random_state=seed),
        binary_classifier=LogisticRegression(
            C=1.5, max_iter=cfg_lr.get('max_iter', 1000),
            solver=cfg_lr.get('solver', 'lbfgs'), class_weight='balanced',
            random_state=seed),
        top2_trigger=True,
        margin_trigger_gap=gap,
        binary_ds_threshold=ds_thr,
    )
    base_mlp = TfidfMLPClassifier(
        n_svd_components=mc.get('n_svd_components', 500),
        hidden_layer_sizes=tuple(mc.get('hidden_layer_sizes', (512, 256))),
        activation='relu',
        alpha=mc.get('alpha', 0.01),
        max_iter=mc.get('max_iter', 300),
        early_stopping=mc.get('early_stopping', True),
        validation_fraction=mc.get('validation_fraction', 0.1),
        random_state=seed,
        deepseek_boost_factor=1.0,
        verbose=False,
    )
    base_lgbm = LGBMTfidfClassifier(
        n_svd_components=lc.get('n_svd_components', 300),
        n_estimators=lc.get('n_estimators', 300),
        num_leaves=lc.get('num_leaves', 31),
        learning_rate=lc.get('learning_rate', 0.05),
        min_child_samples=lc.get('min_child_samples', 10),
        subsample=lc.get('subsample', 0.8),
        colsample_bytree=lc.get('colsample_bytree', 0.8),
        reg_alpha=lc.get('reg_alpha', 0.1),
        reg_lambda=lc.get('reg_lambda', 0.1),
        random_state=seed,
    )
    # Meta-LR: strongly regularized (C=0.1) because 18 meta-features are correlated
    meta_lr = LogisticRegression(C=0.1, max_iter=1000, class_weight='balanced',
                                  random_state=seed)

    return StackingClassifier(
        estimators=[
            ('two_stage_top2', base_two_stage),
            ('mlp_svd',        base_mlp),
            ('lgbm_svd',       base_lgbm),
        ],
        final_estimator=meta_lr,
        cv=3,
        stack_method='predict_proba',
        n_jobs=1,
        passthrough=False,
    )


def build_pipeline(cfg: Dict, seed: int = 42) -> Pipeline:
    """Build full sklearn Pipeline: Preprocessor → FeatureUnion → stacking_lgbm."""
    p = cfg.get('preprocessing', {})
    return Pipeline([
        ('preprocessor', Preprocessor(
            normalize_unicode=p.get('normalize_unicode', True),
            strip_whitespace=p.get('strip_whitespace', True),
            remove_repeated_spaces=p.get('remove_repeated_spaces', True),
            lowercase=p.get('lowercase', False),
            remove_punctuation=p.get('remove_punctuation', False),
            remove_numbers=p.get('remove_numbers', False),
        )),
        ('features',   build_feature_union(cfg)),
        ('classifier', build_stacking_lgbm(cfg, seed)),
    ])


print('Pipeline builder ✓')

## 4. Cross-Validation (5-Fold Stratified)

**Anti-leakage guarantee:** The complete pipeline (Preprocessor → FeatureUnion → Classifier)
is rebuilt and fitted **inside each fold**. TF-IDF vocabulary, IDF weights, stylometric
scaler, and all model weights are computed only on training-fold data. No information
can bleed from validation samples into the feature statistics.

**Expected result:** CV Macro F1 ≈ 0.9393 ± 0.0082  
**Expected time:** 5–15 minutes (hardware-dependent)

In [ ]:
# ── 5-fold Stratified Cross-Validation ────────────────────────────────────────
vc  = CONFIG['validation']
skf = StratifiedKFold(n_splits=vc['n_splits'], shuffle=vc['shuffle'],
                       random_state=vc['random_state'])

X_arr = np.array(X_train, dtype=object)  # array for advanced indexing
fold_scores = []
oof_proba   = np.zeros((len(X_train), N_CLASSES), dtype=np.float64)
oof_preds   = np.zeros(len(X_train), dtype=int)

print(f"Starting {vc['n_splits']}-fold stratified CV (stacking_lgbm)...")
print('(Each fold: 3-fold internal CV inside StackingClassifier + MLP + LGBM)\n')

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_arr, y_train), 1):
    X_tr, X_va = X_arr[tr_idx].tolist(), X_arr[va_idx].tolist()
    y_tr, y_va = y_train[tr_idx], y_train[va_idx]

    # Build + fit fresh pipeline inside this fold — no leakage
    pipe = build_pipeline(CONFIG, seed=RANDOM_STATE)
    pipe.fit(X_tr, y_tr)

    # OOF probability predictions
    fold_proba = pipe.predict_proba(X_va)
    fold_pred  = np.argmax(fold_proba, axis=1)

    oof_proba[va_idx] = fold_proba
    oof_preds[va_idx] = fold_pred

    f1 = f1_score(y_va, fold_pred, average='macro', zero_division=0)
    fold_scores.append(f1)
    print(f'  Fold {fold}/{vc["n_splits"]}  —  Macro F1: {f1:.4f}')

cv_mean = np.mean(fold_scores)
cv_std  = np.std(fold_scores)
oof_f1  = f1_score(y_train, oof_preds, average='macro', zero_division=0)

print(f'\n{"─"*55}')
print(f'CV Macro F1  : {cv_mean:.4f} ± {cv_std:.4f}')
print(f'OOF Macro F1 : {oof_f1:.4f}')
print(f'{"─"*55}')
print(f'\nRun 10 reference: CV = 0.9393, OOF = 0.9397')

In [ ]:
# ── OOF classification report ─────────────────────────────────────────────────
print('OOF Classification Report (before threshold optimization):')
print('─' * 60)
print(classification_report(
    y_train, oof_preds,
    target_names=[CLASS_NAMES[i] for i in range(N_CLASSES)],
    zero_division=0,
))

## 5. Threshold Optimization

Two sequential post-processing steps on out-of-fold predictions:

**Step 1 — Per-class probability scaling** (corrects calibration errors):

$$\hat{y} = \arg\max_{k} \; p_k \cdot s_k$$

Scale factors $s_k$ are grid-searched in $[0.5, 3.0]$ with 11 points per class,
ordered by worst-class-F1-first, with 2 greedy passes.

**Step 2 — DeepSeek/Grok pair ratio threshold** (handles the DS/Grok confusion bottleneck):

$$\text{ratio} = \frac{P_{\text{DS}}}{P_{\text{DS}} + P_{\text{Grok}}}, \quad
\hat{y} = \begin{cases} \text{DeepSeek} & \text{if ratio} \geq \tau \\ \text{Grok} & \text{otherwise} \end{cases}$$

Run 10 optimal values: **scales** = `[2.25, 0.75, 1.0, 1.0, 2.5, 1.75]`, $\tau = 0.55$

In [ ]:
# ── Step 1: optimize per-class probability scales ─────────────────────────────
print('Optimizing per-class probability scales on OOF predictions...')
THRESHOLDS = optimize_thresholds(oof_proba, y_train, n_grid=11, lo=0.5, hi=3.0)
print(f'Optimal per-class scales: {np.round(THRESHOLDS, 3).tolist()}')

oof_preds_t1 = apply_thresholds(oof_proba, THRESHOLDS)
f1_t1 = f1_score(y_train, oof_preds_t1, average='macro', zero_division=0)
print(f'OOF Macro F1 after scaling: {f1_t1:.4f}')

# ── Step 2: optimize DS/Grok pair ratio threshold ─────────────────────────────
print('\nOptimizing DS/Grok pair ratio threshold...')
DS_GROK_THRESHOLD = optimize_ds_grok_threshold(oof_proba, y_train, THRESHOLDS)
print(f'Optimal DS/Grok ratio threshold τ: {DS_GROK_THRESHOLD:.3f}')

# ── Apply both steps ──────────────────────────────────────────────────────────
oof_preds_final = apply_thresholds(oof_proba, THRESHOLDS)
oof_preds_final = apply_ds_grok_pair_threshold(oof_proba, oof_preds_final, DS_GROK_THRESHOLD)
f1_final = f1_score(y_train, oof_preds_final, average='macro', zero_division=0)

print(f'\nOOF Macro F1 after both thresholds: {f1_final:.4f}')
print(f'Run 10 reference: scales=[2.25,0.75,1.0,1.0,2.5,1.75], τ=0.55, OOF=0.9397')

In [ ]:
# ── Post-threshold classification report ──────────────────────────────────────
print('OOF Classification Report (after threshold optimization):')
print('─' * 60)
print(classification_report(
    y_train, oof_preds_final,
    target_names=[CLASS_NAMES[i] for i in range(N_CLASSES)],
    zero_division=0,
))
print()
print('Run 10 reference (OOF after threshold opt):')
print('  Human: 1.00  DeepSeek: 0.77  Grok: 0.90  Claude: 1.00  Gemini: 0.99  ChatGPT: 0.99')

## 6. Final Model Training

Train the `stacking_lgbm` pipeline on the **full 2,400-sample training set**.

The StackingClassifier internally:
1. Trains each of the 3 base estimators using **3-fold cross-validation**
2. Generates 18-dim meta-features (3 models × 6 class probabilities) for all 2,400 samples
3. Fits the meta LogisticRegression on these 18-dim features

**Expected time:** 5–10 minutes

In [ ]:
# ── Train final pipeline on all 2,400 samples ─────────────────────────────────
print('Training final stacking_lgbm on all 2,400 training samples...')
print('(Internal: StackingClassifier runs 3-fold CV for each of 3 base estimators)')
print()

final_pipeline = build_pipeline(CONFIG, seed=RANDOM_STATE)
final_pipeline.fit(X_train, y_train)

# Quick sanity: train-set macro F1 (should be > 0.99)
train_preds = final_pipeline.predict(X_train)
train_f1    = f1_score(y_train, train_preds, average='macro', zero_division=0)
print(f'Train Macro F1 (sanity check): {train_f1:.4f}  (should be > 0.99)')
print('Final model trained ✓')

## 7. Test Prediction & Submission

In [ ]:
# ── Generate test predictions ─────────────────────────────────────────────────
print(f'Generating predictions for {len(X_test)} test samples...')
test_proba = final_pipeline.predict_proba(X_test)
print(f'Probabilities shape: {test_proba.shape}')

# Apply threshold post-processing
test_preds = apply_thresholds(test_proba, THRESHOLDS)
test_preds = apply_ds_grok_pair_threshold(test_proba, test_preds, DS_GROK_THRESHOLD)

# Distribution check
print(f'\nTest prediction distribution:')
for label in range(N_CLASSES):
    count = (test_preds == label).sum()
    pct   = count / len(test_preds) * 100
    bar   = '█' * count
    print(f'  {CLASS_NAMES[label]:>8} [{label}]: {count:4d}  ({pct:.1f}%)  {bar[:40]}')

print()
print('Run 10 reference: {Human:380, DS:23, Grok:38, Claude:20, Gemini:60, ChatGPT:79}')

In [ ]:
# ── Save submission CSV ───────────────────────────────────────────────────────
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
sub_path  = OUTPUT_DIR / f'submission_{timestamp}.csv'

submission = pd.DataFrame({'id': test_ids, 'label': test_preds.astype(int)})
submission.to_csv(sub_path, index=False)

print(f'Submission saved: {sub_path}')
print(f'Shape: {submission.shape}')
print()
print(submission.head(10).to_string(index=False))

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
thresholds_data = {
    'model': 'stacking_lgbm',
    'run': '2026-03-16_221821_run',
    'thresholds': THRESHOLDS.tolist(),
    'ds_grok_pair_threshold': float(DS_GROK_THRESHOLD),
    'cv_macro_f1': float(cv_mean),
    'oof_macro_f1': float(f1_final),
}
thresh_path = OUTPUT_DIR / f'thresholds_{timestamp}.json'
with open(thresh_path, 'w') as f:
    json.dump(thresholds_data, f, indent=2)

print('=' * 62)
print('FINAL SUMMARY — Run 10 Reproduction')
print('=' * 62)
print(f'GitHub    : https://github.com/Sajjad-Shahali/text-authorship-detection')
print(f'Model     : stacking_lgbm')
print(f'          : [TwoStageClassifier + TfidfMLP + LGBMTfidf] → LR meta')
print(f'CV F1     : {cv_mean:.4f} ± {cv_std:.4f}  (5-fold stratified)')
print(f'OOF F1    : {f1_final:.4f}  (after threshold optimization)')
print(f'LB score  : 0.92422  (from original run)')
print(f'Scales    : {np.round(THRESHOLDS, 3).tolist()}')
print(f'DS/Grok τ : {DS_GROK_THRESHOLD:.3f}')
print(f'Submission: {sub_path.name}')
print('=' * 62)

---

## Appendix: Architecture & Design Rationale

### Why Stacking Over Simple Voting?

A soft-voting ensemble uses fixed weights; stacking learns per-model, per-class confidence.
If MLP and LGBM agree on a sample but TwoStage disagrees, the meta-LR has learned
which to trust. This outperforms voting by ~0.004 macro F1 in our experiments.

### The DeepSeek / Grok Bottleneck

Both write short, factual, encyclopedia-style text (topic: science, history, geography).
The key distinguishing features found empirically:

| Feature | DeepSeek | Grok |
|---------|----------|------|
| Sentence length | Uniform, short (5–15 words) | High variance, sometimes long |
| Sentence length CV | Low (< 0.5) | High (> 0.7) |
| Numbered list rate | High | Low |
| Hedging language | Almost none | Moderate |
| Punctuation variety | Low (mostly `.` `:`) | Higher |
| Discourse markers | Rare | Common |

Best run confusion: DS→Grok: 17, Grok→DS: 15 (32 total out of 240 DS+Grok test samples).

### Threshold Interpretation (Run 10)

| Class | Scale | Interpretation |
|-------|------:|----------------|
| Human | 2.25 | Raw proba underestimates Human — boost it |
| DeepSeek | 0.75 | Binary stage over-predicts DS — dampen it |
| Grok | 1.00 | Well-calibrated |
| Claude | 1.00 | Well-calibrated (very distinctive writing) |
| Gemini | 2.50 | Stacking under-recalls Gemini — boost it significantly |
| ChatGPT | 1.75 | Mild recall improvement needed |

### Repository

**https://github.com/Sajjad-Shahali/text-authorship-detection**

```
src/features.py              → feature engineering (cell 5 of this notebook)
src/models.py                → model classes (cell 6)
src/threshold_optimizer.py   → threshold functions (cell 7)
src/train.py                 → CV loop (cell 8)
EXPERIMENTS.md               → full 12-run experiment log
artifacts/experiments/       → frozen artifacts from every run
  2026-03-16_221821_run/     → best run (LB 0.92422)
```